# Senso del Notebook

Questo script è un ottimo **"Sanity Check" (controllo di integrità)**. Serve a verificare che il lungo processo di estrazione delle nuove feature (Step 3) sia andato a buon fine prima di usarle per addestrare la TCN. 

Ecco cosa fa, riga per riga:

1. **Ricerca del path esatto**: Esegue un comando Linux (`find`) per cercare dove si trova esattamente il file `video01_features.npy` all'interno dell'ambiente Kaggle. Serve a confermare che i dati siano stati importati correttamente nel nuovo notebook.
2. **Controllo dei file generati**: Elenca i primi 10 file presenti nella cartella `features_finetuned` per assicurarsi che l'estrazione non si sia interrotta a metà e che i file `.npy` ci siano tutti.
3. **Controllo di qualità sul Video 1**: 
   * Carica i numeri (tensori) del video 1.
   * Ne stampa la forma (ci aspettiamo qualcosa tipo `[numero_frame, 2048]`).
   * Controlla che la *deviazione standard* (`std`) non sia vicina allo zero (se fosse zero, significherebbe che la rete è "collassata" e sputa fuori sempre gli stessi numeri identici per ogni frame, rendendo le feature inutili).
   * Controlla che non ci siano valori `NaN` (Not a Number), che indicherebbero un errore matematico (es. divisione per zero o gradienti esplosi) durante il fine-tuning.
4. **Campionamento incrociato**: Ripete lo stesso controllo di qualità (forma, media, varianza) su un campione di 4 video diversi (1, 15, 45, 80). Prende video sia dall'inizio che dalla fine del dataset per assicurarsi che l'estrazione sia stata uniforme e stabile su tutti gli 80 video, e non si sia corrotta lungo la strada.

In breve: è un test rapido per dirti *"Sì, le tue nuove feature sono sane, contengono informazioni valide e sono pronte per essere date in pasto alla TCN"*!

In [1]:
import subprocess
print(subprocess.run(['find', '/kaggle/input', '-iname', 'video01_features.npy'], 
                      capture_output=True, text=True).stdout)

/kaggle/input/notebooks/christiancorvino/cholec80-resnet50-finetuning/features_finetuned/video01_features.npy



In [2]:
import os
print(os.listdir("/kaggle/input/notebooks/christiancorvino/cholec80-resnet50-finetuning/features_finetuned")[:10])

['video25_labels.npy', 'video66_features.npy', 'video60_labels.npy', 'video45_labels.npy', 'video27_features.npy', 'video12_features.npy', 'video05_labels.npy', 'video57_labels.npy', 'video57_features.npy', 'video39_labels.npy']


In [3]:
import numpy as np
video1 = '/kaggle/input/notebooks/christiancorvino/cholec80-resnet50-finetuning/features_finetuned/video01_features.npy'
f = np.load(video1)
print(f.shape, f.mean(), f.std(), np.isnan(f).any())
# aspettati: std non vicino a 0 (feature non collassate), nessun NaN

(1734, 2048) 0.04371021 0.109069735 False


In [4]:
BASE_FEAT_DIR = "/kaggle/input/notebooks/christiancorvino/cholec80-resnet50-finetuning/features_finetuned"

for vid in [1, 15, 45, 80]:
    f = np.load(f"{BASE_FEAT_DIR}/video{vid:02d}_features.npy")
    print(vid, f.shape, f.mean(), f.std())

1 (1734, 2048) 0.04371021 0.109069735
15 (2059, 2048) 0.048833378 0.12191654
45 (3388, 2048) 0.052468445 0.106591254
80 (1725, 2048) 0.053387392 0.117381245
